# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa – Exploration with `mlcroissant`

This notebook provides an end-to-end guide for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(metadata.name)
print(metadata.description)
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets and their IDs
record_sets = list(dataset.record_sets())
print("Available record sets:")
for rs in record_sets:
    print(f"  - Record Set Name: {rs.name}")
    print(f"    @id: {rs.id}")
    print(f"    Fields:")
    for field in rs.fields:
        print(f"      - Field Name: {field.name}, @id: {field.id}")
    print()

# Print a sample record from each record set using @id
for rs in record_sets:
    print(f"Sample record from record set '{rs.name}' (@id: {rs.id}):")
    records = dataset.records(record_set=rs.id)
    try:
        print(next(records))
    except StopIteration:
        print("No records found.")
    print()

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis.
All references use record set and field `@id`s.

In [ ]:
# Choose the primary survey record set by its @id
# Get all record sets and their ids
record_sets = list(dataset.record_sets())
record_set_ids = [rs.id for rs in record_sets]

# For demo, pick the first record set as main survey set
main_record_set_id = record_set_ids[0]

# Load all records into a DataFrame for each record set
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

print("Column names (fields/@id) in the main record set:")
print(dataframes[main_record_set_id].columns.tolist())

dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming distributions, and grouping data by key attributes. All references use `@id` fields.

In [ ]:
# Example: Analyze GAD-7 score field (assume its @id is 'gad7_score')
# Find numeric field among columns using @id
main_df = dataframes[main_record_set_id]

# If schema provides exact field/order, replace accordingly
# For demo, search for likely GAD-7 score column
numeric_field_ids = [col for col in main_df.columns if 'gad' in col.lower() or 'phq' in col.lower() or 'score' in col.lower()]
print(f"Numeric fields found: {numeric_field_ids}")

# Select the GAD-7 score field by @id
if numeric_field_ids:
    gad7_field_id = numeric_field_ids[0]
else:
    gad7_field_id = main_df.columns[0]

threshold = 10
# Filter records with GAD-7 scores above threshold
filtered_df = main_df[main_df[gad7_field_id] > threshold]
print(f"Filtered records with '{gad7_field_id}' > {threshold}:")
print(filtered_df.head())

# Normalize the GAD-7 score field
norm_col = f"{gad7_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()
print(f"Normalized '{gad7_field_id}' for filtered records:")
print(filtered_df[[gad7_field_id, norm_col]].head())

# Group by a demographic column, e.g., 'gender' or 'age_group' by @id
group_field_id = None
for gf in ['gender', 'age', 'age_group', 'marital_status', 'level_of_education', 'village']:
    for col in main_df.columns:
        if gf in col.lower():
            group_field_id = col
            break
    if group_field_id:
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[gad7_field_id].mean().reset_index()
    print(f"Grouped mean of '{gad7_field_id}' by '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
All visualization refers to fields via their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

main_df = dataframes[main_record_set_id]

# Plot histogram of GAD-7 scores
if gad7_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[gad7_field_id], bins=15, kde=True)
    plt.title(f"Distribution of '{gad7_field_id}' (GAD-7 Scores)")
    plt.xlabel(gad7_field_id)
    plt.ylabel('Frequency')
    plt.show()

# If a group field is available, plot mean scores by group
if group_field_id and gad7_field_id:
    group_means = main_df.groupby(group_field_id)[gad7_field_id].mean().reset_index()
    plt.figure(figsize=(8,4))
    sns.barplot(x=group_field_id, y=gad7_field_id, data=group_means)
    plt.xticks(rotation=45)
    plt.title(f"Mean '{gad7_field_id}' by '{group_field_id}'")
    plt.show()

## 6. Conclusion
This notebook demonstrated loading, overview, extraction, and exploratory analysis of the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`. We reviewed available record sets, accessed their fields via `@id`, performed basic filtering, normalization, grouping, and visualizations.

Key Findings:
* Significant variability in GAD-7 scores is observed.
* Demographic factors (e.g., gender, education) may influence mental health indicators.
* The `mlcroissant` library offers a seamless interface for FAIR and AI-ready exploration.

For further analysis, refer to the dataset's Croissant schema for additional fields and relationships; ensure all access uses entity `@id` for interoperability.